# Concert Program Structured Data with LLM

Extracts place, date, presenting institution, patrons/sponsors, composers and works performed, and performers (with instrument or vocal part) from OCR'd concert program PDFs, tagged with the source page number and merged with the manifest CSV's metadata.

- Notebook by Daniel Russo-Batterham, Richard Freedman, with assistance of Holden Starkey (Haverford College Class of 2027)
- Adapted for concert programs, August 2026.

## 0.  Set up File Paths and Imports

In [1]:
# initial imports of libraries

from pathlib import Path
from langchain_core.documents import Document

from typing import Optional, List

from langchain_core.prompts import ChatPromptTemplate

import asyncio
import getpass
import os
import csv
import json
import re
from datetime import datetime

from pydantic import BaseModel, Field


def load_ocr_txt(filepath: str) -> list:
    """Read a reliable OCR text file and split it into page-like Documents."""
    text = Path(filepath).read_text(encoding="utf-8", errors="replace")
    matches = list(re.finditer(r"^===\s*Page\s+(\d+)\s*===\s*$", text, flags=re.MULTILINE))
    if not matches:
        return [Document(page_content=text.strip())]

    pages = []
    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        page_text = text[start:end].strip()
        if page_text:
            page_number = int(match.group(1))
            pages.append(Document(page_content=page_text, metadata={"page_number": page_number}))
    return pages


def canonical_pdf_name(filename: str) -> str:
    """Normalize a txt or pdf filename to the original PDF ID."""
    if filename.lower().endswith(".pdf"):
        return filename
    if filename.lower().endswith(".txt"):
        return filename.replace(".txt", ".pdf")
    return f"{filename}.pdf"


def canonical_txt_name(filename: str) -> str:
    """Normalize a txt or pdf filename to the OCR text filename."""
    if filename.lower().endswith(".txt"):
        return filename
    if filename.lower().endswith(".pdf"):
        return filename.replace(".pdf", ".txt")
    return f"{filename}.txt"


def resolve_txt_source_path(filename: str) -> Path:
    """Resolve a text-based source file from the OCR manifest and file list."""
    return Path("extracted_texts/OLMOCR") / canonical_txt_name(filename)


def resolve_source_path(filename: str) -> Path:
    """Resolve the preferred source file for a program from the OCR text directory."""
    return resolve_txt_source_path(filename)


async def load_source_documents(filename: str) -> list:
    """Load a source txt file as page-level Documents."""
    source_path = resolve_source_path(filename)
    if source_path.suffix.lower() == ".txt":
        return load_ocr_txt(str(source_path))
    raise FileNotFoundError(f"No .txt source found for {filename} at {source_path}")


def build_marked_up_text(pages: list) -> str:
    """Join per-page Documents into one string, with a literal page marker before each
    page's text. The LLM is asked to copy the nearest preceding marker into page_number,
    rather than compute or guess a page number itself."""
    parts = []
    for idx, page in enumerate(pages, start=1):
        metadata = getattr(page, "metadata", {}) or {}
        page_number = metadata.get("page_number", idx)
        parts.append(f"--- PAGE {page_number} ---\n{page.page_content}")
    return "\n\n".join(parts)


/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# folder containing the OCR text and the manifest describing them
pdf_dir = Path("sources")
manifest_path = Path("extracted_texts/OLMOCR") / "manifest_for_txt.csv"
extracted_text_dir = Path("extracted_texts/OLMOCR")

# each run writes to its own subfolder so earlier results are preserved and never overwritten
output_root = Path("Structured Data")
output_name = "txt_extraction_run"
output_dir = output_root / f"{output_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
counter = 1
while output_dir.exists():
    output_dir = output_root / f"{output_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{counter}"
    counter += 1
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Writing extraction outputs to {output_dir}")

In [22]:

# use a list of filenames here for a quick test run; leave empty to process every row in the
# txt manifest. Filenames may be either .txt or the original PDF name; both forms will resolve.
# Example: sample_filenames = ["UDC20260028-21.txt"]
# sample_filenames: List[str] = ["UDC20260028-21.txt"]
sample_filenames: List[str] = []

# cap on how many NOT-YET-PROCESSED files to extract in a single run (e.g. 10 at a time,
# re-running to work through the manifest in chunks); set to None to process all pending files
batch_size: Optional[int] = None

# how many txt files to send to the API at once within a batch. Higher = faster wall-clock time,
# but pushing this too high risks hitting your OpenAI account's rate limits (429 errors,
# which the client will retry a couple of times before giving up on that file). Start modest
# and raise it if a run completes cleanly with no "Failed to extract" messages.
max_concurrency: int = 4

# organizations to skip entirely (e.g. graphics/score-heavy programs that are slow or
# expensive to process) -- add more names here as needed, matched against the manifest's
# Organization column
skip_organizations: List[str] = ["Cross Talk (in Tokyo)"]

# specific filenames to skip regardless of organization
skip_filenames: List[str] = []


In [13]:
# load the manifest: txt filename -> {contents, date, organization}
# We keep both the txt filename and the original PDF id as aliases so processing can be driven
# by the text manifest while output still carries the original PDF metadata.
manifest = {}
with open(manifest_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        txt_filename = row["Filename"]
        pdf_filename = canonical_pdf_name(txt_filename)
        meta = {
            "contents": row.get("Contents"),
            "date": row.get("Date"),
            "organization": row.get("Organization"),
        }
        manifest[txt_filename] = meta
        manifest[pdf_filename] = meta


print(f"Loaded manifest for {len(manifest)} file entries")

Loaded manifest for 184 file entries


### Select LLM and Provide API Key

- You can choose your preferred LLM here. Make sure to have the necessary API keys set up in your environment.



In [14]:
# This notebook requires an OpenAI API key before any extraction run.
# If OPENAI_API_KEY is not already set in your environment, the prompt below will ask for it.
# Just press Enter after typing the key in the pop-up box.

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

# swap the model here to compare quality/cost/speed on the same sample_filenames.
# GPT-5.6 tiers: "gpt-5.6-sol" (flagship, closest to gpt-5's capability), "gpt-5.6-terra"
# (balanced), "gpt-5.6-luna" (cheapest/fastest, but worth checking it doesn't regress on the
# judgment-call-heavy parts of this task -- OCR-garble handling, patron/performer distinction,
# not fabricating events on non-concert pages -- before trusting it on the full batch).
model_name = "gpt-5"

# a hard cap per request, so a stalled connection raises an error (caught and skipped by the
# extraction loop below) instead of hanging indefinitely. A standalone terminal test measured
# a real 22-page/6-list-schema call at ~265s, so 450s leaves solid headroom above that without
# ballooning the worst-case wait on a genuinely stuck request.
llm = init_chat_model(model_name, model_provider="openai", timeout=450, max_retries=2)


## 1.  Setting up Classes for Pydantic

For concert programs we care about six kinds of item: the venue, the date, the presenting institution/organization, patrons and sponsors, the composers and works performed, and the musicians and other performers (with their instrument or vocal part). Each item is its own Pydantic class, and every item carries a `page_number`, a verbatim `source_text` excerpt, and `review_flags` for anything inferred or uncertain — so every extracted fact can be traced back to exactly where it was found and how confident we are in it.

In [15]:
# Pydantic classes for concert-program structured extraction.
# Every record carries page_number + source_text (verbatim excerpt) + review_flags,
# so each fact can be traced back to where it was found and how confident we are in it.

class Venue(BaseModel):
    """A place where the concert took place (city, hall, building)."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    place: str = Field(description="The place of performance (city, hall, building) as it appears in the text.")

class PerformanceDate(BaseModel):
    """A date on which the concert took place."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    date_text: str = Field(description="The date of performance, verbatim as printed.")

class OrganizationMention(BaseModel):
    """An institution or organization presenting or performing the concert."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    name: str = Field(description="The organization's name as it appears in the text.")
    role: Optional[str] = Field(default=None, description="The organization's role, e.g. 'presenting organization', 'choir', 'orchestra'.")

class Patron(BaseModel):
    """A patron or sponsor named in the program, distinct from the performing organization."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    name: str = Field(description="The patron or sponsor's name as it appears in the text.")
    role: Optional[str] = Field(default=None, description="Their role, e.g. Patron, Vice-Patron, Sponsor, Donor, President.")

class WorkPerformed(BaseModel):
    """A composed work performed at the concert."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    composer: Optional[str] = Field(default=None, description="The work's composer, if given.")
    title: str = Field(description="The title of the work performed.")
    movement_or_selection: Optional[str] = Field(default=None, description="A specific movement or selection from the work, if indicated.")

class Performer(BaseModel):
    """A musician or other performer, and what they played, sang, or their role."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    name: str = Field(description="The performer's name as it appears in the text.")
    part_or_instrument: Optional[str] = Field(default=None, description="The instrument played, vocal part sung, or role (e.g. 'soprano', 'violin', 'conductor', 'accompanist').")
    associated_work: Optional[str] = Field(default=None, description="The title of the work they are credited on, if clear.")

class ConcertProgramExtraction(BaseModel):
    """Everything extracted from one concert program text file."""
    source_pdf: str = Field(default="", description="Original PDF id that corresponds to the OCR text file.")
    txt_source_file: str = Field(default="", description="Relative path to the OCR text file used for extraction.")
    venues: List[Venue] = Field(default_factory=list)
    dates: List[PerformanceDate] = Field(default_factory=list)
    organizations: List[OrganizationMention] = Field(default_factory=list)
    patrons: List[Patron] = Field(default_factory=list)
    works: List[WorkPerformed] = Field(default_factory=list)
    performers: List[Performer] = Field(default_factory=list)


In [16]:
def flatten_extraction(filename: str, meta: dict, extraction_dict: dict) -> List[dict]:
    """Turn one concert's ConcertProgramExtraction (as a plain dict) into flat item records,
    each stamped with its record_type, original PDF id, txt source path, and manifest metadata."""
    items = []
    source_pdf = extraction_dict.get("source_pdf") or canonical_pdf_name(filename)
    txt_source_file = extraction_dict.get("txt_source_file") or str(resolve_source_path(filename))
    for record_type, key in [
        ("venue", "venues"),
        ("date", "dates"),
        ("organization", "organizations"),
        ("patron", "patrons"),
        ("work", "works"),
        ("performer", "performers"),
    ]:
        for record in extraction_dict.get(key, []):
            items.append({
                "record_type": record_type,
                "filename": filename,
                "source_pdf": source_pdf,
                "txt_source_file": txt_source_file,
                "manifest_contents": meta.get("contents"),
                "manifest_date": meta.get("date"),
                "manifest_organization": meta.get("organization"),
                **record,
            })
    return items


## 2.  Bind the Schema to the LLM

`ConcertProgramExtraction` bundles all six item lists for one concert program. We bind it to the LLM's structured-output mode so a single call returns validated Pydantic objects rather than raw JSON.

In [17]:
structured_llm = llm.with_structured_output(ConcertProgramExtraction)

## 3.  The System Prompt

In [18]:
# Common part/role abbreviations found in these historical programs -- mostly Italian ordinal
# conventions for orchestral sections. Add more entries here as you find them across the corpus;
# they're automatically folded into the system prompt below, no prompt-editing required.
#
# "Imi."/"I mi" = Primi ("First"), "IIdi." = Secondi ("Second"), "IIIzi." = Terzi ("Third")
role_glossary = {
    "Violini Imi.": "First Violin",
    "Violini IIdi.": "Second Violin",
    "Viole": "Viola",
    "Violoncelli": "Violoncello (Cello)",
    "Contrabassi": "Double Bass",
    "Flauti Imi.": "First Flute",
    "Flauti IIdi.": "Second Flute",
    "Oboi": "Oboe",
    "Clarinetti": "Clarinet",
    "Fagotti": "Bassoon",
    "Corni": "Horn",
    "Trombe": "Trumpet",
    "Tromboni": "Trombone",
    "Timpani": "Timpani",
}

In [19]:
# system and human prompts for extracting concert-program data -- may need adjustment for different printers/OCR quality

role_glossary_text = "\n".join(f'- "{k}" = {v}' for k, v in role_glossary.items())

system_prompt = f"""
You are a musicologist and archivist specializing in 19th- and 20th-century concert programs.
You will be given the full text of one concert program, extracted via OCR from a PDF. The text
has literal page markers of the form "--- PAGE n ---" inserted before each page's content.

Your job is to extract every mention of the following, across all pages:
- Place of performance (city, hall, building)
- Date of performance
- The institution or organization presenting/performing the concert (e.g. a choral society,
  orchestra, or academy)
- Patrons and sponsors -- people or bodies named as patron, vice-patron, sponsor, donor, or
  similar honorary/financial role. These are usually distinct from the performing organization
  itself (e.g. a Governor or Lord Mayor listed as "Patron" of a concert given by the
  "Melbourne Liedertafel").
- Composers and the works they performed
- Musicians and other performers, and what instrument they played or vocal part they sang
  (or their role, such as conductor or accompanist)

Rules:
- For every item you extract, set page_number to the number copied from the nearest preceding
  "--- PAGE n ---" marker. Never guess or compute a page number -- only copy it from a marker.
- For every item, set source_text to a short verbatim excerpt of the OCR text that supports it.
  Do not paraphrase or correct the wording in source_text.
- Name and title fields (composer, title, and every person/organization name) must always hold
  the text exactly as printed/OCR'd -- never silently correct or normalize a spelling. If the
  OCR is garbled but you can confidently guess the correct reading (e.g. "Geethoven" for
  "Beethoven"), keep the garbled form in the field itself and add a review_flags note giving
  your best-guess correction. Only use null when the text is too garbled to make any confident
  guess at all, and flag it as unrecoverable.
- A performer's part_or_instrument is different: it is a controlled-vocabulary field, so
  normalize it to a clear, standard English description rather than reproducing the printed
  abbreviation. Many of these programs abbreviate orchestral sections using Italian ordinal
  conventions: "Imi."/"I mi" = Primi ("First"), "IIdi." = Secondi ("Second"), "IIIzi." = Terzi
  ("Third"). For example "Violini Imi." means First Violin, "Corni IIdi." means Second Horn.
  Known examples from this corpus:
{role_glossary_text}
  Apply the same ordinal pattern to any instrument section you recognize it on, even if it is
  not in the list above. If you encounter a role/instrument abbreviation you do not recognize at
  all, keep it as printed in part_or_instrument and add a review_flags note rather than guessing.
- Some pages are not part of the concert program itself (e.g. a travel itinerary, an
  institutional history/prospectus, or garbled/illegible OCR noise). Such pages should simply
  yield no items -- do not invent a place, date, work, or performer that isn't there.
- Whenever you infer a value, expand an abbreviated name, guess at OCR-garbled text, or make an
  ambiguous attribution (for example, a role caption that isn't clearly tied to one name), add a
  short reason to that item's review_flags. Prefer leaving a field null and flagging it over
  fabricating a value.
- If a work lists several movements or several composers, emit one work item per distinct work.
- If a performer is clearly credited on a specific work, set that performer's associated_work to
  that work's title.
"""

human_prompt = """
Given the following concert program (with page markers), extract every venue, date,
organization, patron/sponsor, work performed, and performer as a structured
ConcertProgramExtraction object.
"""

## 4.  Extract Data from Each Concert Program

Each PDF is short enough to send in a single LLM call: we join its pages into one string with literal `--- PAGE n ---` markers and let the model copy the right marker into each item's `page_number`, which also lets it correlate a work on one page with performers listed on the next. Since there's only one call per document now (no scene-by-scene chunking), we don't need a LangGraph graph -- a plain function does the job.

This step **resumes** rather than starting over: it loads any previously saved `concert_program_by_file.json` as a cache and only calls the LLM for filenames not already in it. So if you add new rows to `manifest.csv` later, re-running this notebook only pays for the new files -- it won't re-extract ones already done. To force a specific file to be redone (e.g. after a prompt change), add its filename to `force_reprocess_filenames` below.

It also runs up to `max_concurrency` files **in parallel** rather than one at a time, since each call is just waiting on the network -- a batch of 10 no longer has to wait for each file to finish before starting the next. Each file still succeeds or fails independently (one slow/failing file doesn't block or cancel the others), and progress prints as each one completes.

**This cell saves its own progress after every file**, so it's self-contained: for your *next* batch, just re-run this cell on its own -- no need to run Section 5 in between to avoid losing progress. Section 5 is still what regenerates the flat JSON/CSV *views* (`concert_program_items.json`/`.csv`); run it whenever you want those refreshed, but it's no longer required between extraction batches.

In [20]:
# don't edit these

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Context:\n{context}\n\nQuestion:\n{question}")
])

async def extract_concert(filename: str) -> ConcertProgramExtraction:
    """Load one concert program text file and run structured extraction."""
    source_path = resolve_source_path(filename)
    pages = await load_source_documents(filename)
    marked_up_text = build_marked_up_text(pages)
    message = prompt.invoke({"question": human_prompt, "context": marked_up_text})
    # Runs the *synchronous* client in a worker thread rather than using
    # structured_llm.ainvoke() directly -- a standalone terminal test confirmed the sync
    # client cleanly raises APITimeoutError at the configured timeout, while the async
    # client's timeout enforcement under Jupyter's event loop is unverified and may be why
    # earlier runs hung silently for 100+ minutes instead of failing after ~15 min.
    extraction = await asyncio.to_thread(structured_llm.invoke, message)
    pdf_name = canonical_pdf_name(filename)
    return extraction.model_copy(update={
        "source_pdf": pdf_name,
        "txt_source_file": str(source_path),
    })


In [23]:
# resume support: reuse any previously extracted results so re-running after adding new
# files to the manifest doesn't re-call the LLM (and re-pay for) files already processed
by_file_json_path = output_dir / "concert_program_by_file.json"
if by_file_json_path.exists():
    with open(by_file_json_path, encoding="utf-8") as f:
        by_file: dict = json.load(f)
    print(f"Loaded {len(by_file)} previously processed files from {by_file_json_path}")
else:
    by_file: dict = {}


def save_by_file():
    """Persist by_file to disk immediately -- called after every successful extraction so
    progress is never lost to an interrupted/hung run, and this cell is safe to just re-run
    on its own for the next batch without needing Section 5 in between."""
    with open(by_file_json_path, "w", encoding="utf-8") as f:
        json.dump(by_file, f, indent=2, ensure_ascii=False)


# list filenames here to force them to be re-extracted even if already cached (e.g. after a
# prompt change). Accept either .txt, .pdf, or a bare root ID and normalize automatically.
force_reprocess_filenames: List[str] = []

requested_filenames = sample_filenames if sample_filenames else list(manifest.keys())
filenames_to_process = []
seen = set()
for name in requested_filenames:
    normalized = canonical_pdf_name(name)
    if normalized not in seen:
        filenames_to_process.append(normalized)
        seen.add(normalized)

skipped_filenames = [
    f for f in filenames_to_process
    if canonical_pdf_name(f) in skip_filenames or manifest.get(f, {}).get("organization") in skip_organizations
]
filenames_to_process = [f for f in filenames_to_process if f not in skipped_filenames]
if skipped_filenames:
    print(f"Skipping {len(skipped_filenames)} files per skip_organizations/skip_filenames: {skipped_filenames}")

pending_filenames = [
    f for f in filenames_to_process
    if f not in by_file or canonical_pdf_name(f) in force_reprocess_filenames
]
candidate_filenames = pending_filenames if batch_size is None else pending_filenames[:batch_size]

# validate before dispatching -- skip anything missing from the manifest or without a usable txt source
filenames_to_extract = []
for filename in candidate_filenames:
    if manifest.get(filename) is None and manifest.get(canonical_txt_name(filename)) is None:
        print(f"Skipping {filename}: not found in manifest")
        continue

    source_path = resolve_source_path(filename)
    if not source_path.exists():
        print(f"Skipping {filename}: no txt source found at {source_path}")
        continue
    filenames_to_extract.append(filename)

print(f"{len(filenames_to_process) - len(pending_filenames)} already cached, "
      f"{len(pending_filenames)} pending, extracting {len(filenames_to_extract)} this run "
      f"(up to {max_concurrency} at a time) from the .txt manifest")

# run extractions concurrently, bounded by max_concurrency, so files don't wait on each other.
# Each file's success/failure is handled independently -- one slow or failing file doesn't
# block or cancel the rest of the batch.
semaphore = asyncio.Semaphore(max_concurrency)


async def bounded_extract(filename: str):
    async with semaphore:
        try:
            extraction = await extract_concert(filename)
            return filename, extraction, None
        except Exception as e:
            return filename, None, e


tasks = [asyncio.create_task(bounded_extract(filename)) for filename in filenames_to_extract]
for finished in asyncio.as_completed(tasks):
    filename, extraction, error = await finished
    if error is not None:
        print(f"Failed to extract {filename}: {error}")
        continue
    by_file[filename] = extraction.model_dump()
    save_by_file()
    print(f"Extracted {filename} from {canonical_txt_name(filename)}")

print(f"by_file now has {len(by_file)} processed files total")


Loaded 20 previously processed files from Structured Data/txt_extraction_run_20260902_183516/concert_program_by_file.json
Skipping 3 files per skip_organizations/skip_filenames: ['UDC20260028-12.pdf', 'UDC20260028-5.pdf', 'UDC20260028-7.pdf']
Skipping filenames..pdf: no txt source found at extracted_texts/OLMOCR/filenames..txt
20 already cached, 69 pending, extracting 68 this run (up to 4 at a time) from the .txt manifest
Extracted UDC20260028-29.pdf from UDC20260028-29.txt
Extracted UDC20260028-30.pdf from UDC20260028-30.txt
Extracted UDC20260028-31.pdf from UDC20260028-31.txt
Extracted UDC20260028-32.pdf from UDC20260028-32.txt
Extracted UDC20260028-33.pdf from UDC20260028-33.txt
Extracted UDC20260028-34.pdf from UDC20260028-34.txt
Extracted UDC20260028-36.pdf from UDC20260028-36.txt
Extracted UDC20260028-3.pdf from UDC20260028-3.txt
Extracted UDC20260028-35.pdf from UDC20260028-35.txt
Extracted UDC20260028-37.pdf from UDC20260028-37.txt
Extracted UDC20260028-38.pdf from UDC20260028-

## 5.  Output

`concert_program_by_file.json` is already kept up to date by Section 4 (saved after every file). This section derives the other two views from it -- run it whenever you want them refreshed, not necessarily after every batch:

- `concert_program_items.json` -- a flat JSON array, one object per extracted item (venue, date, organization, patron, work, or performer), each carrying its `record_type`, source `filename`, the manifest metadata (Contents/Date/Organization), and the `page_number` it was found on.
- `concert_program_by_file.json` -- the same data grouped by source file.
- `concert_program_items.csv` -- a flat CSV mirror of the item list, for spreadsheet use.

In [ ]:
# flatten every processed file (cached + newly extracted) into item records
all_items: List[dict] = []
for filename, extraction_dict in by_file.items():
    meta = manifest.get(filename)
    if meta is None:
        continue  # file no longer listed in the manifest
    all_items.extend(flatten_extraction(filename, meta, extraction_dict))

# flat JSON: one record per extracted item (venue, date, organization, patron, work, performer)
flat_json_path = output_dir / "concert_program_items.json"
with open(flat_json_path, "w", encoding="utf-8") as f:
    json.dump(all_items, f, indent=2, ensure_ascii=False)

# nested JSON: grouped by source file -- also the resume cache read by the extraction step
with open(by_file_json_path, "w", encoding="utf-8") as f:
    json.dump(by_file, f, indent=2, ensure_ascii=False)

# flat CSV mirror of the item list
csv_path = output_dir / "concert_program_items.csv"
fieldnames = sorted({key for item in all_items for key in item.keys()})
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for item in all_items:
        row = dict(item)
        if isinstance(row.get("review_flags"), list):
            row["review_flags"] = "; ".join(row["review_flags"])
        writer.writerow(row)

print(f"Wrote {len(all_items)} items to {flat_json_path}")
print(f"Wrote {len(by_file)} files to {by_file_json_path}")
print(f"Wrote {len(all_items)} rows to {csv_path}")